# Energy inputs

Check the inputs of the energy build: the CEB data, the OpenStreetMap files and the VIIRS night lights. Each section prints counts, shows a table and draws a map. This notebook builds and downloads nothing; if a file is missing, the cell prints the command that makes it.

| Input | Made by | Location |
|---|---|---|
| CEB data (not public) | shared by CEB | `data/incoming/Infrastructure/Energy/`: `Substation`, `Power Transmission`, `Generation Source`, `Power Demand` |
| CEB Annual Report tables | typed in from the 2023-24 report | `data/incoming/Infrastructure/Energy/CEB Annual Report/` |
| Cleaned CEB data | rule `prepare_energy_assets` | `data/processed/energy/<model_data>/provided/` |
| OpenStreetMap files | rule `fetch_energy_osm`, when a file is missing | `data/incoming/Infrastructure/Energy/OpenStreetMap/<region>/` |
| VIIRS monthly images | rule `fetch_energy_nightlights`, when an image is missing | `data/incoming/Infrastructure/Energy/Nighttime Lights/viirs-2024-monthly/` |
| WorldPop population, for the demand shares | rule `fetch_energy_population`, when the file is missing | `data/incoming/Infrastructure/Energy/Population/` |

`<model_data>` and `<region>` are `model_data` and `region` in `config/energy/energy.yaml`. "Running the model" in `docs/src/infrastructure-energy.md` says how to get the data. To build and inspect the networks, go on to `01_networks.ipynb`.

In [ ]:
import pathlib
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers finds the workflow's files and draws them.
import _helpers as h  # noqa: E402

print("data root: ", h.DATA_ROOT)
print("model data:", h.MODEL_DATA)
print("region:    ", h.REGION)

## CEB data

The `prepare_energy_assets` rule cleans CEB's shapefiles and demand workbook and reads the CEB Annual Report tables. It writes to `data/processed/energy/<model_data>/provided/`:

- `substations`, and `snapped_substations` moved onto the nearest 66 kV route (`substation_snap_distances.csv` has the distance moved);
- `transmission_routes`: the 66 kV routes, with voltage and length where the source gives them;
- `generation_points` and `generation_areas`: CEB's generation sites as drawn, and `generation_sites.csv`: the same sites as points, with their nearest substation;
- `generators.csv`: one row per plant in the CEB Annual Report 2023-24, on Mauritius and Rodrigues, with its type (`carrier`), installed capacity (`output_capacity_mw`) and the CEB substation it is assigned to (`bus_id`). `bus_id` is empty for plants on Rodrigues, which has no CEB substation in the data, for distributed generation such as rooftop solar (SSDG, MSDG), and for plants with no location yet;
- `monthly_peak_demand_mw.csv` and `annual_sector_demand_gwh.csv`, from the demand workbook;
- `service_weights.csv`: an equal share of demand for each substation, a placeholder. The estimated shares come from the `build_energy_demand` rule.

The rule needs the CEB data in `data/incoming/Infrastructure/Energy/`.

In [ ]:
provided = h.load_provided() if h.found(h.PROVIDED_DIR / "generators.csv", h.PREPARE_PROVIDED) else {}
for name, table in provided.items():
    print(f"{name:30s} {len(table):5d} rows   {type(table).__name__}")

In [ ]:
if h.found(h.PROVIDED_DIR / "generators.csv", h.PREPARE_PROVIDED):
    generators = provided["generators"]
    plants = gpd.GeoDataFrame(
        generators, geometry=gpd.points_from_xy(generators["lon"], generators["lat"]), crs="EPSG:4326"
    )
    h.quick_map(
        transmission_routes=provided["transmission_routes"],
        substations_as_supplied=(provided["substations"], {"color": "0.4", "markersize": 16}),
        substations_snapped_to_route=provided["snapped_substations"],
        plants=(plants, {"marker": "^", "markersize": 50}),
        clip=h.MAURITIUS_BBOX,
        title="CEB data, Mauritius: 66 kV routes, substations and plants",
    )

In [ ]:
if h.found(h.PROVIDED_DIR / "generators.csv", h.PREPARE_PROVIDED):
    generators = provided["generators"]
    print("Plants by island (with_bus_id: assigned to a CEB substation):")
    by_island = generators.assign(with_bus_id=generators["bus_id"].notna()).groupby("region")
    display(
        by_island.agg(
            plants=("generator_id", "size"),
            output_capacity_mw=("output_capacity_mw", "sum"),
            with_bus_id=("with_bus_id", "sum"),
        ).round(1)
    )
    columns = ["generator_id", "name", "region", "carrier", "output_capacity_mw", "bus_id", "bus_assignment_distance_m"]
    display(generators[columns].sort_values("output_capacity_mw", ascending=False).reset_index(drop=True))

In [ ]:
if h.found(h.PROVIDED_DIR / "monthly_peak_demand_mw.csv", h.PREPARE_PROVIDED):
    monthly_peak = provided["monthly_peak_demand_mw"].set_index("year")
    annual = provided["annual_sector_demand_gwh"].pivot(index="year", columns="category", values="demand_gwh")
    print("Monthly system peak demand (MW), latest years:")
    display(monthly_peak.tail())
    print("Annual electricity demand by customer sector (GWh):")
    display(annual.round(1))
    ax = monthly_peak.T.plot(figsize=(9, 4), cmap="viridis", legend=False)
    ax.set_title("Monthly system peak demand (MW), one line per year, purple (first year) to yellow (last year)")
    ax.set_xlabel("month")

## OpenStreetMap files

The inferred networks need three OpenStreetMap layers for `energy.region` (`mauritius-rodrigues` is downloaded island by island and joined):

- `roads.parquet`: drivable roads (`energy.osm.network_type: drive`), one row per street with its `highway` class. The estimated distribution lines follow these roads.
- `power.parquet`: mapped substations, plants and generators as points. They are the power assets of `inferred-osm`, and some CEB plants are placed by their name here.
- `aoi.parquet`: island outlines from Nominatim, used to clip the night-light image.

They are in `data/incoming/Infrastructure/Energy/OpenStreetMap/<region>/`. The `fetch_energy_osm` rule downloads them when a file is missing.

In [ ]:
osm_files = h.osm_paths()
osm = h.load_osm_cache()
for key, path in osm_files.items():
    if h.found(path, h.FETCH_OSM):
        print(f"{key:6s} {len(osm[key]):6d} features   {path}")
if "roads" in osm:
    print("\nRoads by highway class and region:")
    by_class = osm["roads"].groupby(["highway", "region"], dropna=False).size().unstack("region", fill_value=0)
    display(by_class.assign(total=by_class.sum(axis=1)).sort_values("total", ascending=False))
if "power" in osm:
    print("Power features by kind and region:")
    display(osm["power"].groupby(["power", "region"]).size().unstack("region", fill_value=0))

In [ ]:
if h.found(osm_files["roads"], h.FETCH_OSM):
    power = osm.get("power")
    by_kind = (
        {} if power is None else {kind: power[power["power"].eq(kind)] for kind in ("substation", "plant", "generator")}
    )
    h.quick_map(
        roads=(osm["roads"], {"color": "0.75", "linewidth": 0.3}),
        area_of_interest=(osm.get("aoi"), {"color": "black", "linewidth": 1.2}),
        substations=(by_kind.get("substation"), {"marker": "s", "markersize": 60}),
        plants=(by_kind.get("plant"), {"marker": "^", "markersize": 40}),
        generators=(by_kind.get("generator"), {"markersize": 10}),
        clip=h.MAURITIUS_BBOX,
        title="OpenStreetMap cache, Mauritius: drivable roads, power features and the area-of-interest outline",
    )

## Night lights

The inferred networks keep roads within `energy.nightlight.nightlight_support_distance_m` (1 km) of a lit pixel or a power asset. The lit pixels come from three rules:

1. `fetch_energy_nightlights` downloads the twelve monthly VIIRS images of 2024 to `data/incoming/Infrastructure/Energy/Nighttime Lights/viirs-2024-monthly/` (`NN-<object id>.tif`).
2. `build_energy_nightlight_composite` takes the median of each pixel: `data/processed/energy/<model_data>/nightlight/<region>/viirs-composite.tif`.
3. `build_energy_nightlight_targets` applies the GridFinder filter and threshold (`energy.nightlight.nightlight_threshold`) inside the island outlines and writes the lit pixels as points, `targets.geoparquet`, with a `targets.tif` mask and `metadata.json`.

`snakemake -c1 build_energy_networks` runs steps 2 and 3; the commands printed below run them on their own.

In [ ]:
tiles = h.nightlight_monthly_tiles()
cached = [tile for tile in tiles if tile.is_file()]
print(f"{len(cached)} of {len(tiles)} monthly VIIRS tiles cached in {h.NIGHTLIGHT_MONTHLY_DIR}")
if len(cached) < len(tiles):
    print("  missing tiles -- run:", h.FETCH_NIGHTLIGHTS)

nightlight = h.nightlight_paths()
if h.found(nightlight["composite"], h.BUILD_NIGHTLIGHT_COMPOSITE):
    composite = rioxarray.open_rasterio(nightlight["composite"], masked=True).squeeze()
    bounds = tuple(round(value, 3) for value in composite.rio.bounds())
    print(f"composite: {composite.rio.width} x {composite.rio.height} pixels, {composite.rio.crs}, bounds {bounds}")
    fig, ax = plt.subplots(figsize=(9, 7))
    composite.rio.clip_box(*h.MAURITIUS_BBOX).plot.imshow(ax=ax, robust=True, cmap="magma")
    ax.set_title("VIIRS 2024 median radiance, Mauritius (use h.RODRIGUES_BBOX for Rodrigues)")

In [ ]:
if h.found(nightlight["targets"], h.BUILD_NIGHTLIGHT_TARGETS):
    targets = h.load_nightlight_targets()
    radiance = targets["radiance"]
    print(f"{len(targets)} night-light targets, radiance {radiance.min():.2f} to {radiance.max():.2f}")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), width_ratios=[3, 1])
    for ax, (island, bbox) in zip(axes, (("Mauritius", h.MAURITIUS_BBOX), ("Rodrigues", h.RODRIGUES_BBOX))):
        on_island = targets.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]]
        h.quick_map(
            area_of_interest=(osm.get("aoi"), {"color": "black"}),
            targets=(targets, {"markersize": 4, "color": "darkorange"}),
            clip=bbox,
            ax=ax,
            title=f"{island}: {len(on_island)} targets",
        )

if h.found(nightlight["metadata"], h.BUILD_NIGHTLIGHT_TARGETS):
    print(nightlight["metadata"].read_text())

## Next

Build the networks and inspect them in `01_networks.ipynb` (`snakemake -c1 build_energy_networks` in a terminal, or `RUN_BUILD = True` in that notebook).